# Run a classifer on the branchwater search output

In [1]:
# base checkout of workflow directory is here:
BASE='/home/ctbrown/scratch3/2025-workflow-core99/'

# parquet files from branchwater
#BASE_OUTPUTS=BASE+'/outputs.branchwater'

SCALED=1000
HASH_THRESHOLD=0 #int(30000 / SCALED)

#BASE_OUTPUTS=BASE+f'/outputs.mapping/cds/singlehash.k21/outputs.branchwater.min{MIN}.scaled{SCALED}'
BASE_OUTPUTS=BASE+'/outputs.cds/branchwater.cds3-genes/'

In [2]:
import polars as pl
import numpy as np
import sklearn.tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

In [3]:
dirpath = BASE_OUTPUTS+'/*.parquet'

bw_df = pl.scan_parquet(dirpath).collect()
bw_df = bw_df.with_columns(pl.col("query_name").alias("species"),
                           pl.col("match_name").alias("acc"))
bw_df = bw_df.select(["species", "acc", "containment", "intersect_hashes"])

# mimic the Web site search threshold of 0.1
bw_df = bw_df.filter(pl.col("intersect_hashes") >= HASH_THRESHOLD)
bw_df

species,acc,containment,intersect_hashes
str,str,f64,i64
"""s__Bariatricus sp004560705""","""SRR26820262""",1.0,6
"""s__Bariatricus sp004560705""","""SRR26820201""",1.0,6
"""s__Bariatricus sp004560705""","""ERR1135245""",1.0,6
"""s__Bariatricus sp004560705""","""SRR17211190""",1.0,6
"""s__Bariatricus sp004560705""","""SRR9866673""",1.0,6
…,…,…,…
"""s__Sodaliphilus sp004557565""","""SRR23018249""",0.142857,1
"""s__Sodaliphilus sp004557565""","""ERR6910995""",0.142857,1
"""s__Sodaliphilus sp004557565""","""SRR28061934""",0.142857,1


In [4]:
metadata_df = pl.scan_parquet('/group/ctbrowngrp5/sra-metagenomes/20241128-metadata.parquet')
metadata_df = metadata_df.select(["acc", "organism", "assay_type"])\
    .filter(pl.col("acc") != "NP")\
    .filter(pl.col("assay_type") == "WGS")

metadata_df = metadata_df.collect()
metadata_df

acc,organism,assay_type
str,str,str
"""SRR28523869""","""human metagenome""","""WGS"""
"""SRR19901137""","""Streptococcus suis""","""WGS"""
"""SRR24962475""","""Escherichia coli""","""WGS"""
"""SRR29161330""","""biofilm metagenome""","""WGS"""
"""SRR26668739""","""bovine gut metagenome""","""WGS"""
…,…,…
"""ERR4174288""","""human gut metagenome""","""WGS"""
"""ERR3160109""","""human gut metagenome""","""WGS"""
"""ERR2709457""","""human gut metagenome""","""WGS"""


In [5]:
bw_df = bw_df.join(metadata_df, on="acc", how="left")
bw_df

species,acc,containment,intersect_hashes,organism,assay_type
str,str,f64,i64,str,str
"""s__Bariatricus sp004560705""","""SRR26820262""",1.0,6,"""pig gut metagenome""","""WGS"""
"""s__Bariatricus sp004560705""","""SRR26820201""",1.0,6,"""pig gut metagenome""","""WGS"""
"""s__Bariatricus sp004560705""","""ERR1135245""",1.0,6,"""pig gut metagenome""","""WGS"""
"""s__Bariatricus sp004560705""","""SRR17211190""",1.0,6,null,null
"""s__Bariatricus sp004560705""","""SRR9866673""",1.0,6,"""Lawsonia intracellularis""","""WGS"""
…,…,…,…,…,…
"""s__Sodaliphilus sp004557565""","""SRR23018249""",0.142857,1,"""metagenome""","""WGS"""
"""s__Sodaliphilus sp004557565""","""ERR6910995""",0.142857,1,"""Homo sapiens""","""WGS"""
"""s__Sodaliphilus sp004557565""","""SRR28061934""",0.142857,1,"""soil metagenome""","""WGS"""


In [6]:
bw_df['organism'].value_counts().sort(by='count', descending=True)

organism,count
str,u32
"""human gut metagenome""",86849
"""pig gut metagenome""",46305
null,32600
"""gut metagenome""",24955
"""metagenome""",23190
…,…
"""Alyxoria varia""",1
"""Brevundimonas subvibrioides""",1
"""lichen crust metagenome""",1


In [7]:
rename_df = []
for organism in bw_df['organism'].unique().to_list():
    simpleorg = 'unknown'
    if organism:
        organism = organism.lower()
        for kw in ['human', 'homo']:
            if kw in organism.lower():
                simpleorg = 'human'
                break
        for kw in ['pig', 'sus', 'scrofa']:
            if kw in organism.lower():
                simpleorg = 'pig'
                break
    rename_df.append(dict(organism=organism, simpleorg=simpleorg))

rename_df = pl.DataFrame(rename_df)
rename_df['simpleorg'].value_counts().sort(by='count', descending=True)


simpleorg,count
str,u32
"""unknown""",577
"""human""",24
"""pig""",9


In [8]:
rename_df

organism,simpleorg
str,str
"""environmental samples""","""unknown"""
"""uncultured acidobacteriia bact…","""unknown"""
"""fermentation metagenome""","""unknown"""
"""planctomycetota bacterium""","""unknown"""
"""cladonia furcata""","""unknown"""
…,…
"""microcoleus anatoxicus ptrs2""","""unknown"""
"""rinodina peloleuca""","""unknown"""
"""alistipes""","""unknown"""


In [9]:
# add a new column with a simplified organism
bw_df = bw_df.join(rename_df, on='organism', how='inner', coalesce=True).select(['acc', 'organism', 'species', 'simpleorg'])
bw_df.select(['acc', 'simpleorg']).unique()['simpleorg'].value_counts()


simpleorg,count
str,u32
"""unknown""",66998
"""pig""",6834
"""human""",60782


In [10]:
bw_df

acc,organism,species,simpleorg
str,str,str,str
"""SRR26820262""","""pig gut metagenome""","""s__Bariatricus sp004560705""","""pig"""
"""SRR26820201""","""pig gut metagenome""","""s__Bariatricus sp004560705""","""pig"""
"""ERR1135245""","""pig gut metagenome""","""s__Bariatricus sp004560705""","""pig"""
"""ERR1135286""","""pig gut metagenome""","""s__Bariatricus sp004560705""","""pig"""
"""SRR26064500""","""pig gut metagenome""","""s__Bariatricus sp004560705""","""pig"""
…,…,…,…
"""SRR2199652""","""gut metagenome""","""s__Sodaliphilus sp004557565""","""unknown"""
"""SRR23018249""","""metagenome""","""s__Sodaliphilus sp004557565""","""unknown"""
"""SRR28061934""","""soil metagenome""","""s__Sodaliphilus sp004557565""","""unknown"""


In [11]:
def make_matrices(df):
    acc_df = df['acc'].unique().to_frame().with_row_index(name='acc_index')
    species_df = df['species'].unique().to_frame().with_row_index(name='species_index')
    org_df = df['simpleorg'].unique().to_frame().with_row_index(name='org_index')

    df = df.join(org_df, on='simpleorg', how='left')
    df = df.join(acc_df, on='acc', how='left')
    df = df.join(species_df, on='species', how='left')

    obs = np.zeros((len(acc_df), len(species_df)))
    target = np.zeros((len(acc_df)))

    for row in df.iter_rows(named=True):
        acc_id = row["acc_index"]
        org_id = row["org_index"]
        species_id = row["species_index"]

        obs[acc_id, species_id] = 1
        target[acc_id] = org_id

    print(f'observations matrix shape is: {obs.shape}')

    return df, obs, target


In [12]:
human_pig_only = bw_df.filter(pl.col("simpleorg") != "unknown")
hp_df, hp_obs, hp_target = make_matrices(human_pig_only)

observations matrix shape is: (67616, 10)


In [13]:
hp_df

acc,organism,species,simpleorg,org_index,acc_index,species_index
str,str,str,str,u32,u32,u32
"""SRR26820262""","""pig gut metagenome""","""s__Bariatricus sp004560705""","""pig""",1,33381,2
"""SRR26820201""","""pig gut metagenome""","""s__Bariatricus sp004560705""","""pig""",1,39625,2
"""ERR1135245""","""pig gut metagenome""","""s__Bariatricus sp004560705""","""pig""",1,1831,2
"""ERR1135286""","""pig gut metagenome""","""s__Bariatricus sp004560705""","""pig""",1,59947,2
"""SRR26064500""","""pig gut metagenome""","""s__Bariatricus sp004560705""","""pig""",1,64150,2
…,…,…,…,…,…,…
"""ERR9190661""","""human gut metagenome""","""s__Sodaliphilus sp004557565""","""human""",0,468,3
"""ERR1297591""","""human gut metagenome""","""s__Sodaliphilus sp004557565""","""human""",0,44123,3
"""ERR7477300""","""human gut metagenome""","""s__Sodaliphilus sp004557565""","""human""",0,17606,3


## 6-Fold Cross Validation approach

Run 6 different splits of the data; train on 1/6 of the data and test on the other 5/6s of the data, and then repeat that for each of the 6 splits.

In [14]:
kf = StratifiedKFold(n_splits=6)

accuracies = []
i = 1
for train_sub, test_sub in kf.split(hp_obs, hp_target):
    dt = DecisionTreeClassifier(random_state=42)
    tree = dt.fit(hp_obs[train_sub], hp_target[train_sub])
    pred = tree.predict(hp_obs[test_sub])

    accuracy = balanced_accuracy_score(hp_target[test_sub], pred)
    accuracies.append(accuracy)
    print(f"iteration {i}: accuracy {accuracy:.3f}")
    i += 1

print(f"mean accuracy across {i-1} splits: {np.mean(accuracies):.3f}")

iteration 1: accuracy 0.936
iteration 2: accuracy 0.941
iteration 3: accuracy 0.943
iteration 4: accuracy 0.941
iteration 5: accuracy 0.946
iteration 6: accuracy 0.939
mean accuracy across 6 splits: 0.941


## What do we after 6-fold cross validation?

This approach lets us check to see that, in general, the performance of any classifer trained on this data will be pretty good - independent of which set of data we use.

Now, to train the best possible classifier for _future_ use, we will use all the data to train:

In [15]:
dt = DecisionTreeClassifier(random_state=42)
tree = dt.fit(hp_obs, hp_target)
pred = tree.predict(hp_obs)

accuracy = balanced_accuracy_score(hp_target, pred)
print(f"full classifier: accuracy {accuracy:.3f}")

full classifier: accuracy 0.945


## What do we do with the full classifier now??

Now we can go back to the original data - not just the human/pig subset, but the one with unknowns. (Or, potentially, new data sets.)

The only thing we need to be careful about is to encode new observations using the species indexes that match the trained classifier, as well as the organism indexes (0 pig, 1 human). Let's extract those:

In [16]:
species_index_df = hp_df.select(['species_index', 'species']).unique()
species_index_df

species_index,species
u32,str
7,"""s__Holdemanella porci"""
5,"""s__JALFVM01 sp022787145"""
3,"""s__Sodaliphilus sp004557565"""
4,"""s__Phascolarctobacterium_A suc…"
2,"""s__Bariatricus sp004560705"""
8,"""s__Prevotella sp002251295"""
9,"""s__Lactobacillus amylovorus"""
6,"""s__Cryptobacteroides sp9005469…"
1,"""s__Mogibacterium_A kristiansen…"


In [17]:
org_df = hp_df.select(['org_index', 'simpleorg']).unique()
org_df

org_index,simpleorg
u32,str
0,"""human"""
1,"""pig"""


Now, let's take some of some of the data we didn't touch from the original bw_df, and encode and classify it: 

In [18]:
def make_obs_matrix(df, species_df):
    # need a new access mapping
    acc_df = df['acc'].unique().to_frame().with_row_index(name='acc_index')

    # construct new internal data frame, populate
    df = df.join(acc_df, on='acc', how='left')
    df = df.join(species_df, on='species', how='left')

    obs = np.zeros((len(acc_df), len(species_df)))

    for row in df.iter_rows(named=True):
        acc_id = row["acc_index"]
        species_id = row["species_index"]

        obs[acc_id, species_id] = 1

    print(f'observations matrix shape is: {obs.shape}')

    return df, obs


### First, do pig only; how many do we lose??

In [19]:
pig_only_df = bw_df.filter(pl.col("simpleorg") == "pig")
pig_df, pig_obs = make_obs_matrix(pig_only_df, species_index_df)

pig_pred = tree.predict(pig_obs)

observations matrix shape is: (6834, 10)


In [20]:
org_df

org_index,simpleorg
u32,str
0,"""human"""
1,"""pig"""


In [21]:
sum(pig_pred == org_df.filter(pl.col('simpleorg') == "pig")['org_index'].to_list()[0]) / len(pig_pred)

np.float64(0.8943517705589699)

### Now, unknown.

In [22]:
unknown_only_df = bw_df.filter(pl.col("simpleorg") == "unknown")
unk_df, unk_obs = make_obs_matrix(unknown_only_df, species_index_df)

observations matrix shape is: (66998, 10)


In [23]:
unknown_pred = tree.predict(unk_obs)
unknown_pred

array([0., 0., 0., ..., 0., 0., 0.], shape=(66998,))

## Predictions are great and all but...

Great! XXX predictions 😭 . What do we do with them??

Let's merge the predictions back into the spreadsheet.

In [24]:
unk_pred_df = []
for (row_n, predicted_org) in enumerate(unknown_pred):
    unk_pred_df.append(dict(acc_index=row_n, org_index=int(predicted_org)))
unk_pred_df = pl.DataFrame(unk_pred_df).join(org_df, on='org_index', how='left')

unk_acc_df = unk_df.select(["acc", "acc_index"]).unique()

unk_pred_df = unk_pred_df.join(unk_acc_df, on='acc_index', how='left')
unk_pred_df

acc_index,org_index,simpleorg,acc
i64,i64,str,str
0,0,"""human""","""SRR6323447"""
1,0,"""human""","""SRR28022630"""
2,0,"""human""","""SRR30610595"""
3,0,"""human""","""SRR2937351"""
4,0,"""human""","""SRR20217209"""
…,…,…,…
66993,0,"""human""","""SRR17191899"""
66994,0,"""human""","""SRR29934347"""
66995,0,"""human""","""ERR3094294"""


In [25]:
unk_pred_df["simpleorg"].value_counts().sort(by='count', descending=True)

simpleorg,count
str,u32
"""human""",63054
"""pig""",3944


## OK! We have our predictions... did this actually work?

In [26]:
# sample 5 randomly
unk_pred_df.sample(n=5)

acc_index,org_index,simpleorg,acc
i64,i64,str,str
51993,0,"""human""","""ERR4368539"""
4607,0,"""human""","""SRR14517522"""
55425,0,"""human""","""ERR2750816"""
27036,0,"""human""","""DRR526072"""
51154,0,"""human""","""SRR10983026"""


In [27]:
# sample 5 predicted as pig
unk_pred_df.filter(pl.col("simpleorg") == "pig").sample(n=5)

acc_index,org_index,simpleorg,acc
i64,i64,str,str
17923,1,"""pig""","""SRR30201829"""
29123,1,"""pig""","""ERR6397220"""
33032,1,"""pig""","""SRR18183268"""
39616,1,"""pig""","""SRR25322310"""
17254,1,"""pig""","""ERR2241785"""


In [28]:
# sample 5 predicted as human
unk_pred_df.filter(pl.col("simpleorg") == "human").sample(n=5)

acc_index,org_index,simpleorg,acc
i64,i64,str,str
27411,0,"""human""","""SRR8552912"""
14606,0,"""human""","""SRR16971107"""
39352,0,"""human""","""SRR25607933"""
60924,0,"""human""","""SRR27973524"""
10856,0,"""human""","""SRR28426649"""


# What features are most important for good classification?

First, let's get the feature importances.

In [29]:
importances_df = []
for (species_index, importance) in enumerate(tree.feature_importances_):
    d = dict(species_index=species_index, importance=importance)
    importances_df.append(d)

importances_df = pl.DataFrame(importances_df)
importances_df = importances_df.join(species_index_df, on='species_index', how='left')
importances_df = importances_df.sort(by='importance', descending=True)
importances_df

species_index,importance,species
i64,f64,str
9,0.753443,"""s__Lactobacillus amylovorus"""
3,0.098355,"""s__Sodaliphilus sp004557565"""
8,0.090997,"""s__Prevotella sp002251295"""
2,0.025382,"""s__Bariatricus sp004560705"""
5,0.010016,"""s__JALFVM01 sp022787145"""
1,0.006092,"""s__Mogibacterium_A kristiansen…"
4,0.005515,"""s__Phascolarctobacterium_A suc…"
7,0.003801,"""s__Holdemanella porci"""
0,0.003199,"""s__Prevotella sp000434975"""


Now let's rebuild the classifier after removing each species, and see how well it does...

In [30]:
species_in_order = importances_df['species'].to_list()
species_in_order

['s__Lactobacillus amylovorus',
 's__Sodaliphilus sp004557565',
 's__Prevotella sp002251295',
 's__Bariatricus sp004560705',
 's__JALFVM01 sp022787145',
 's__Mogibacterium_A kristiansenii',
 's__Phascolarctobacterium_A succinatutens',
 's__Holdemanella porci',
 's__Prevotella sp000434975',
 's__Cryptobacteroides sp900546925']

In [31]:
human_pig_only = bw_df.filter(pl.col("simpleorg") != "unknown")
species_to_rm = list(species_in_order)

remove_df = human_pig_only
while len(species_to_rm) >= 2:
    rm_species = species_to_rm.pop(0)
    print(f'removing {rm_species} too')
    if len(species_to_rm) == 1:
        print('left:', species_to_rm[0])

    remove_df = remove_df.filter(pl.col("species") != rm_species)
    augmented_df, rm_obs, rm_target = make_matrices(remove_df)

    dt = DecisionTreeClassifier(random_state=42)
    tree = dt.fit(rm_obs, rm_target)
    pred = tree.predict(rm_obs)

    accuracy = balanced_accuracy_score(rm_target, pred)
    print(f"classifier accuracy with {remove_df['species'].n_unique()} is: {accuracy:.3f}")
    print('---')

removing s__Lactobacillus amylovorus too
observations matrix shape is: (66854, 9)
classifier accuracy with 9 is: 0.930
---
removing s__Sodaliphilus sp004557565 too
observations matrix shape is: (65114, 8)
classifier accuracy with 8 is: 0.919
---
removing s__Prevotella sp002251295 too
observations matrix shape is: (64898, 7)
classifier accuracy with 7 is: 0.884
---
removing s__Bariatricus sp004560705 too
observations matrix shape is: (56283, 6)
classifier accuracy with 6 is: 0.834
---
removing s__JALFVM01 sp022787145 too
observations matrix shape is: (56263, 5)
classifier accuracy with 5 is: 0.845
---
removing s__Mogibacterium_A kristiansenii too
observations matrix shape is: (54496, 4)
classifier accuracy with 4 is: 0.780
---
removing s__Phascolarctobacterium_A succinatutens too
observations matrix shape is: (47913, 3)
classifier accuracy with 3 is: 0.819
---
removing s__Holdemanella porci too
observations matrix shape is: (33907, 2)
classifier accuracy with 2 is: 0.817
---
removing s_

## Remove in reverse order...

In [32]:
human_pig_only = bw_df.filter(pl.col("simpleorg") != "unknown")
species_to_rm = list(reversed(species_in_order))

remove_df = human_pig_only
while len(species_to_rm) >= 2:
    rm_species = species_to_rm.pop(0)
    print(f'removing {rm_species} too')

    remove_df = remove_df.filter(pl.col("species") != rm_species)
    augmented_df, rm_obs, rm_target = make_matrices(remove_df)

    dt = DecisionTreeClassifier(random_state=42)
    tree = dt.fit(rm_obs, rm_target)
    pred = tree.predict(rm_obs)

    accuracy = balanced_accuracy_score(rm_target, pred)
    print(f"classifier accuracy with {remove_df['species'].n_unique()} is: {accuracy:.3f}")
    print('---')

removing s__Cryptobacteroides sp900546925 too
observations matrix shape is: (60144, 9)
classifier accuracy with 9 is: 0.943
---
removing s__Prevotella sp000434975 too
observations matrix shape is: (54333, 8)
classifier accuracy with 8 is: 0.947
---
removing s__Holdemanella porci too
observations matrix shape is: (41776, 7)
classifier accuracy with 7 is: 0.941
---
removing s__Phascolarctobacterium_A succinatutens too
observations matrix shape is: (32581, 6)
classifier accuracy with 6 is: 0.945
---
removing s__Mogibacterium_A kristiansenii too
observations matrix shape is: (28426, 5)
classifier accuracy with 5 is: 0.944
---
removing s__JALFVM01 sp022787145 too
observations matrix shape is: (28418, 4)
classifier accuracy with 4 is: 0.940
---
removing s__Bariatricus sp004560705 too
observations matrix shape is: (14704, 3)
classifier accuracy with 3 is: 0.915
---
removing s__Prevotella sp002251295 too
observations matrix shape is: (14302, 2)
classifier accuracy with 2 is: 0.888
---
removing